# RoboManipBaselines Colab Tutorial

## Getting Started with Manipulation × AI：模倣学習の体験セッション

このノートブックは、[SIマニピュレーション若手の会](https://sites.google.com/view/sice-si-manipulation/Wakate) がワークショップ用に作成した、Google Colab向けのハンズオン教材です。

この教材の目的は、ロボット学習や深層学習にまだ慣れていない方でも、**「模倣学習では何を準備し、何を学習し、どのような結果が得られるのか」** を一通り体験できるようにすることです。数式や論文の詳細に入る前に、まずは実際に動くコードを上から順に実行し、全体の流れをつかむことを重視しています。

題材には、産業技術総合研究所を中心に開発されているオープンソースソフトウェア [RoboManipBaselines](https://github.com/isri-aist/RoboManipBaselines) を使用します。RoboManipBaselinesは、ロボットマニピュレーションにおける模倣学習手法を、実世界およびシミュレーション環境で再現・検証するためのフレームワークです。

### このノートブックで体験すること

1. Google Colabの実行環境を確認する
2. RoboManipBaselinesをインストールする
3. ACTという模倣学習手法を使うための追加依存関係を入れる
4. サンプルのロボット操作データセットをダウンロードする
5. データセットを使ってACTポリシーを学習する

### 用語の簡単な説明

| 用語 | この教材での意味 |
| --- | --- |
| 模倣学習 | 人や既存コントローラが行った操作データをまねるように、ロボットの行動を学習する方法 |
| デモンストレーション | お手本となるロボット操作の記録。画像、関節角、手先位置、行動などが含まれます |
| ポリシー | 観測を入力として、次にロボットが取るべき行動を出力するモデル |
| ACT | Action Chunking with Transformerの略。複数ステップ分の行動をまとめて予測する模倣学習手法 |
| checkpoint | 学習済みモデルの重みを保存したファイル。あとでrolloutや評価に使います |

<br>

> **注意**  
> このノートブックはワークショップ用の非公式教材です。RoboManipBaselinesの開発者および公式リポジトリに、このノートブック固有の内容について問い合わせないでください。

関連リンク:

- [RoboManipBaselines GitHub Repository](https://github.com/isri-aist/RoboManipBaselines)
- [RoboManipBaselines Project Page](https://isri-aist.github.io/RoboManipBaselines-ProjectPage/)
- [RoboManipBaselines arXiv](https://arxiv.org/abs/2509.17057)


## 0. Google Colabの設定

このノートブックは、Google ColabのGPUランタイムで実行することを想定しています。

1. 上部メニューから **ランタイム** → **ランタイムのタイプを変更** を開く
2. **ハードウェア アクセラレータ** に **T4 GPU** または利用可能なGPUを選択する
3. **保存** を押す
4. 上から順にセルを実行する

最初のコードセルでは、現在のPython環境とGPUの状態を確認します。これは「これから学習を実行できる状態か」を見るための準備です。

### ここで確認するポイント

- `Python:` にPythonのバージョンが表示される
- `nvidia-smi` の出力にGPU名が表示される
- T4 GPUなどのGPUが表示されれば、深層学習の計算にGPUを使える可能性が高い

GPUが表示されない場合でも、すぐに失敗とは限りません。まずはランタイム設定を確認してください。Colabの計算資源は保証されず、利用状況によりGPUの種類や利用可能時間が変わることがあります。


In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
except Exception as exc:
    print("GPUを確認できませんでした。ランタイムのハードウェアアクセラレータをGPUに変更してください。")
    print(type(exc).__name__, exc)


## 1. RoboManipBaselinesのインストール

<img src="https://isri-aist.github.io/RoboManipBaselines-ProjectPage/media/images/logo.png" alt="RoboManipBaselines overview" width="20%">

このセルでは、RoboManipBaselines本体と、それに必要な外部ライブラリをColab環境にインストールします。

Colabは毎回まっさらな仮想マシンとして起動するため、ノートブックを開くたびに必要なライブラリを入れ直す必要があります。ここでは `/content/RoboManipBaselines` にGitHubリポジトリを取得し、Pythonから使えるようにします。

### このセルで行っていること

| 処理 | 意味 |
| --- | --- |
| `git clone ... --recursive` | RoboManipBaselines本体とサブモジュールをまとめて取得します |
| `apt-get install ffmpeg` | 動画や画像列を扱うための外部コマンドを入れます |
| `pip install torch...` | 深層学習ライブラリPyTorchを、CUDA 12.4向けの版に揃えます |
| `pip install torchcodec` | 動画データを扱うためのライブラリを入れます |
| `pip install -e .` | RoboManipBaselinesをPythonパッケージとして使えるようにします |

`-e` はeditable installの意味です。通常の利用では細かく意識しなくて構いませんが、取得したリポジトリ内のコードをそのままPythonから参照できるインストール方法です。

### 時間がかかる理由

このセルはリポジトリ取得、依存ライブラリのインストール、PyTorchの入れ替えを行うため、数分かかることがあります。赤いエラーで止まらずに最後まで進めば問題ありません。


In [ ]:
%%bash
set -euxo pipefail

cd /content

if [ ! -d RoboManipBaselines/.git ]; then
  git clone https://github.com/isri-aist/RoboManipBaselines.git --recursive
else
  cd /content/RoboManipBaselines
  git pull --ff-only
  git submodule update --init --recursive
fi

apt-get update -qq
apt-get install -y -qq ffmpeg

python -m pip install --upgrade pip setuptools wheel
python -m pip uninstall -y torch torchvision torchaudio torchcodec || true
python -m pip install \
  torch==2.6.0+cu124 \
  torchvision==0.21.0+cu124 \
  torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124
python -m pip install torchcodec==0.2.0

cd /content/RoboManipBaselines
python -m pip install -e .


## 2. ACT用の追加依存関係をインストール

このチュートリアルでは、模倣学習手法の一つである **Action Chunking with Transformer (ACT)** を使います。

ACTは、カメラ画像やロボットの状態を入力として、ロボットが次に取るべき行動を予測するモデルです。特徴的なのは、1ステップずつ行動を出すのではなく、**複数ステップ分の行動列をまとめて予測する** 点です。これにより、ロボット操作のような滑らかな時系列行動を扱いやすくなります。

### ACTを直感的に理解する

ロボット操作では、ある瞬間の画像だけを見て「次の1コマだけ」動かすよりも、少し先まで見越して一連の動作を出した方が自然な場合があります。例えば、ケーブルをつかんで動かす操作では、

1. ケーブルの端に近づく
2. グリッパを閉じる
3. ケーブルを引く
4. 目標位置へ動かす

というように、複数の動作が連続しています。

ACTでは、このような連続動作を **action chunk**、つまり「行動のまとまり」として予測します。1時刻ごとに細かく迷いながら動くのではなく、短い行動列をまとめて出すことで、操作全体を安定させやすくします。

### この教材でのACTの入出力

このハンズオンでは、おおまかに以下のような対応を学習します。

| 種類 | 例 | 役割 |
| --- | --- | --- |
| 入力 | カメラ画像 | ロボットの周囲や対象物の状態を見る |
| 入力 | 関節角などのロボット状態 | 現在のロボット姿勢を知る |
| 出力 | 関節指令などの行動 | 次にロボットをどう動かすかを決める |

つまりACTは、

> 「今見えている画像」と「今のロボット状態」から、「このあと数ステップで実行する行動列」を予測するモデル

として使われます。

### なぜTransformerを使うのか

ACTの内部ではTransformer系のモデルが使われます。Transformerは、時系列データや画像特徴の中で「どの情報に注目するか」を扱いやすいモデルです。

ただし、このワークショップではTransformerの数式を理解する必要はありません。まずは、

- 画像と状態を入力にする
- 行動列を出力する
- お手本データに近づくように学習する

という3点を押さえれば十分です。

<img src="https://tonyzhaozh.github.io/aloha/resources/algo.png" alt="ACTの概念図" width="70%">

### action chunkとtemporal ensembling

ACTでは、モデルが複数ステップ分の行動をまとめて予測します。この「まとめて予測する長さ」を **chunk size** と呼びます。
初心者向けには、次のように理解しておけば十分です。

| 概念 | 簡単な意味 |
| --- | --- |
| action chunk | 数ステップ分の行動をまとめたもの |
| chunk size | まとめて予測する行動列の長さ |

### ここで入れるもの

このセルでは、ACTを動かすために必要な追加パッケージをインストールします。

- RoboManipBaselinesのACT用追加パッケージ
- ACTの内部で使われる `detr` というTransformer系モデルの実装

TransformerやDETRの詳細を理解していなくても、このハンズオンは進められます。ここでは「ACTを動かすために必要な部品を追加で入れている」と考えてください。


In [ ]:
%%bash
set -euxo pipefail

cd /content/RoboManipBaselines
python -m pip install -e ".[act]"

cd /content/RoboManipBaselines/third_party/act/detr
python -m pip install -e .


## 3. インストール結果を確認

ここでは、インストールしたPyTorchがGPUを認識しているかを確認します。

`torch.cuda.is_available()` が `True` になっていれば、PyTorchからGPUを使える状態です。GPUが使えると、深層学習モデルの学習がCPUより大幅に速くなります。

### 出力の見方

| 表示 | 見るポイント |
| --- | --- |
| `torch:` | PyTorchのバージョンです。ここでは `2.6.0+cu124` に近い表示を期待します |
| `CUDA available:` | `True` ならGPUが使えます。`False` ならランタイム設定を確認します |
| `CUDA device:` | 利用中のGPU名です。T4などが表示されます |
| `CUDA version used by PyTorch:` | PyTorchが使うCUDAのバージョンです |

もし `CUDA available: False` になった場合は、上部メニューのランタイム設定でGPUが選択されているか確認し、必要に応じてランタイムを再起動してください。


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA version used by PyTorch:", torch.version.cuda)


## 4. サンプルデータセットのダウンロード

学習の流れを体験するために、`MujocoUR5eCable_Dataset30` をダウンロードします。このデータセットには、`MujocoUR5eCable` 環境でケーブルを操作するデモンストレーションが30本含まれています。

模倣学習では、モデルに「正解の行動」を教えるためのお手本データが必要です。このお手本データがデモンストレーションです。ロボット操作の場合、単なる画像だけでなく、関節角、手先位置、グリッパの状態、行動コマンドなどが時系列で保存されます。

### このセルで行っていること

1. データセット保存先へ移動する
2. Dropboxからzipファイルをダウンロードする
3. `MujocoUR5eCable` ディレクトリに展開する
4. 不要になったzipファイルを削除する
5. データサイズとファイルの一部を表示する

データセットは `/content/RoboManipBaselines/robo_manip_baselines/dataset/MujocoUR5eCable` に展開します。再実行時に古い展開結果が残らないよう、展開先を一度削除してから作成します。

### 補足: なぜデータをダウンロードするのか

本来は、人がロボットを操作してデータを集めます。しかしワークショップやColab環境では実機ロボットや操作デバイスを用意しにくいため、公開済みのデータセットを使って学習部分を体験します。


In [ ]:
%%bash
set -euxo pipefail

DATASET_ROOT=/content/RoboManipBaselines/robo_manip_baselines/dataset
DATASET_ZIP=MujocoUR5eCable_Dataset30.zip
DATASET_DIR=MujocoUR5eCable
DATASET_URL="https://www.dropbox.com/scl/fo/sykc20cnax2scom1u8sc6/AM-zLM8dAZ5h6EQ8eDXcZic?rlkey=7icbmjc6wdqnp0tngfjqlhwoh&dl=1"

cd "${DATASET_ROOT}"
rm -rf "${DATASET_DIR}"
mkdir -p "${DATASET_DIR}"

curl -L -o "${DATASET_ZIP}" "${DATASET_URL}"
unzip -q -o "${DATASET_ZIP}" -d "${DATASET_DIR}"
rm -f "${DATASET_ZIP}"

du -sh "${DATASET_DIR}"
find "${DATASET_DIR}" -maxdepth 2 -type f | head


## 5. ACTを学習

ダウンロードしたデータセットを使って、ACTの学習を実行します。

ここで行う学習は、「観測から行動を予測するモデル」をデモンストレーションデータに合わせて調整する処理です。入力にはカメラ画像やロボット状態が使われ、出力にはロボットの関節指令などが使われます。

### 実行されるコマンド

```bash
python ./bin/Train.py Act --dataset_dir ./dataset/MujocoUR5eCable --batch_size 32 --num_epochs 1000
```

| 引数 | 意味 |
| --- | --- |
| `Act` | 使う学習手法としてACTを指定します。他にDiffusionPolicyなどがあります。 |
| `--dataset_dir` | 学習に使うデータセットの場所を指定します。 |
| `--batch_size 32` | 1回の更新でまとめて処理するデータ数を指定します。 |
| `--num_epochs 1000` | 学習を繰り返す回数を指定します。Epochとは1回の学習データ全体を処理する単位です。 |


### ログで見るポイント

学習が始まると、以下のような情報が表示されます。

- データセットの読み込み結果
- train / validationのデータ数
- モデルのパラメータ数
- epochごとの進捗
- lossの値
- checkpointの保存先

`loss` は、モデルの予測とお手本データのずれを表す指標です。一般には小さいほどよいですが、ワークショップではまず「学習が始まり、checkpointが保存される」ことを確認できれば十分です。

### 時間について

学習には時間がかかります。ワークショップ中は、ログが流れ始め、損失値や保存先が確認できれば、必要に応じて途中で停止して構いません。最後まで実行した場合、`checkpoint/Act/...` 以下に学習済みモデルが保存されます。今回は--num_epochs 1000を指定していますが、全体で30-40分程度かかる見込みです。途中で停止してもcheckpointは保存されるため、学習の途中経過を後でrolloutや評価に使うことができます。


In [ ]:
%cd /content/RoboManipBaselines/robo_manip_baselines
!python ./bin/Train.py Act --dataset_dir ./dataset/MujocoUR5eCable --batch_size 32 --num_epochs 1000

## 6. 次に確認すること

学習が開始できたら、以下を確認します。

- 学習ログに表示されるlossの推移
- 学習済みモデルやログの保存先
- データセットの件数、画像、ロボット状態、行動の対応関係
- 実機やシミュレーションで使う場合に追加で必要になる設定

### 学習結果はどこに保存されるか

RoboManipBaselinesでは、通常 `robo_manip_baselines/checkpoint/Act/` 以下に学習結果が保存されます。中には、以下のようなファイルが作られます。

| ファイル | 役割 |
| --- | --- |
| `policy_last.ckpt` | 最後のepochのモデル重み |
| `policy_best.ckpt` | validation lossが最もよかったモデル重み |
| `model_meta_info.pkl` | 学習時のデータ構造や正規化情報など、rolloutに必要なメタ情報 |

### このあと何をするか

このノートブックでは、模倣学習の「学習」までを扱いました。次の段階では、学習済みモデルを使ってシミュレーション環境内でロボットを動かす **rollout** を行います。

このリポジトリの `02_rmb_colab_rollout.ipynb` では、学習済みcheckpointを使って実際に動作を生成し、その結果をMP4動画として可視化します。ColabではGUI表示に制約があるため、動画として保存して確認する方針を取っています。

### よくあるつまずき

| 状況 | 対応 |
| --- | --- |
| GPUが使えない | ランタイム設定でGPUを選び、ランタイムを再起動します |
| インストールが途中で止まる | セルを再実行します。依存関係の一時的なダウンロード失敗で止まることがあります |
| 学習が長い | ワークショップでは途中停止しても構いません。公開済みcheckpointを使ったrolloutも可能です |
| Colabが切断された | Colabのセッションは一定時間で切断されます。必要な結果はGoogle Driveやローカル環境に保存してください |
